# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset Croissant package
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets and their fields using @id references
print("Available Record Sets:\n-----------------------")
record_sets = getattr(metadata, 'recordSet', [])
for rec in record_sets:
    rec_id = getattr(rec, '@id', None)
    rec_name = getattr(rec, 'name', None)
    print(f"Record Set @id: {rec_id}, name: {rec_name}")
    rec_fields = getattr(rec, 'field', [])
    print("  Fields:")
    for fld in rec_fields:
        fid = getattr(fld, '@id', None)
        fname = getattr(fld, 'name', None)
        label = getattr(fld, 'label', None)
        print(f"    Field @id: {fid}, name: {fname}, label: {label}")
    print("")

# View example records for the first record set (if any)
if record_sets:
    rs0_id = getattr(record_sets[0], '@id', None)
    print(f"First 2 records from Record Set @id: {rs0_id}")
    for idx, rec in enumerate(dataset.records(record_set=rs0_id)):
        print(rec)
        if idx >= 1:
            break
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all available record sets into DataFrames
# Reference everything by @id, and store in dictionary {record_set_id: DataFrame}
df_by_id = {}
record_set_ids = []

for rec in getattr(metadata, 'recordSet', []):
    rec_id = getattr(rec, '@id', None)
    record_set_ids.append(rec_id)
    rec_records = list(dataset.records(record_set=rec_id))
    df_by_id[rec_id] = pd.DataFrame(rec_records)

if record_set_ids:
    print("Record Set @ids available:")
    for i, rsid in enumerate(record_set_ids, 1):
        print(f"  {i}. {rsid}")
    print("")
    # Show columns of the first record set
    _sample_rs = record_set_ids[0]
    print(f"Fields (columns) for '{@id}={_sample_rs}':\n{df_by_id[_sample_rs].columns.tolist()}")
    display(df_by_id[_sample_rs].head())
else:
    print("No record sets with data available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set and a numeric field by @id for EDA
# Find the first numeric field in the first record set
import numpy as np

if record_set_ids:
    rs_eda = record_set_ids[0]
    rec = next((x for x in getattr(metadata, 'recordSet', []) if getattr(x, '@id', None) == rs_eda), None)
    numeric_field_id = None
    group_field_id = None

    # Try to find a float or integer field for numeric analysis
    for fld in getattr(rec, 'field', []):
        ftype = getattr(fld, 'dataType', None)
        if ftype and ("Float" in ftype or "Integer" in ftype or "Number" in ftype):
            numeric_field_id = getattr(fld, '@id', None)
            break
    # Try to find a field suitable for grouping
    for fld in getattr(rec, 'field', []):
        if getattr(fld, '@id', None) != numeric_field_id and getattr(fld, 'dataType', None) == 'Text':
            group_field_id = getattr(fld, '@id', None)
            break

    df = df_by_id[rs_eda]
    if numeric_field_id and numeric_field_id in df.columns:
        print(f"Using numeric field @id: {numeric_field_id}")
        # Coerce to float for processing
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].mean()  # filter on above-mean value
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows\n")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group and aggregate
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df)
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field is available, show mean by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.barplot(data=df.groupby(group_field_id)[numeric_field_id].mean().reset_index(),
                    x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: no suitable numeric/group field.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*This notebook demonstrated how to load, explore, and analyze the FAIR^2 mass spectrometry dataset using the `mlcroissant` library and referenced all data elements by their `@id`, as required by the Croissant schema. You can adapt this workflow to other Croissant datasets and expand the EDA/visualization as needed for specific research questions.*